In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               AdaBoostClassifier)
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, roc_curve, confusion_matrix, classification_report)

from deployment.Data_prep import build_preprocessors, load_and_prepare


RANDOM_STATE = 42

In [26]:
X, y = load_and_prepare("Data\loan_data.csv")
print("X shape:", X.shape)
print("Columns:", X.columns.tolist())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, " Test:", X_test.shape)

linear_preprocessor, tree_preprocessor = build_preprocessors()

X shape: (45000, 16)
Columns: ['person_age', 'person_gender', 'person_education', 'person_income', 'person_emp_exp', 'person_home_ownership', 'loan_amnt', 'loan_intent', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score', 'previous_loan_defaults_on_file', 'loan_burden', 'total_interest_cost', 'income_per_exp_year']
Train: (36000, 16)  Test: (9000, 16)


In [27]:
counts = y.value_counts()
ratios = y.value_counts(normalize=True) * 100
print(counts)
print(ratios.round(2))

loan_status
0    35000
1    10000
Name: count, dtype: int64
loan_status
0    77.78
1    22.22
Name: proportion, dtype: float64


In [28]:
def evaluate_quick(pipeline, X_tr, y_tr, X_te, y_te, label):
    pipeline.fit(X_tr, y_tr)
    preds = pipeline.predict(X_te)
    proba = pipeline.predict_proba(X_te)[:, 1]
    return {
        "Strategy": label,
        "Accuracy": accuracy_score(y_te, preds),
        "Recall (Approved)": recall_score(y_te, preds),
        "Precision (Approved)": precision_score(y_te, preds),
        "F1 (Approved)": f1_score(y_te, preds),
        "ROC-AUC": roc_auc_score(y_te, proba),
    }


imbalance_comparison = []

# --- Logistic Regression ---
pipe_baseline_lr = ImbPipeline([("prep", linear_preprocessor),
                                 ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
imbalance_comparison.append({"Model": "Logistic Regression",
                              **evaluate_quick(pipe_baseline_lr, X_train, y_train, X_test, y_test, "Baseline")})

pipe_cw_lr = ImbPipeline([("prep", linear_preprocessor),
                           ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE))])
imbalance_comparison.append({"Model": "Logistic Regression",
                              **evaluate_quick(pipe_cw_lr, X_train, y_train, X_test, y_test, "class_weight='balanced'")})

pipe_smote_lr = ImbPipeline([("prep", linear_preprocessor),
                              ("smote", SMOTE(random_state=RANDOM_STATE)),
                              ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
imbalance_comparison.append({"Model": "Logistic Regression",
                              **evaluate_quick(pipe_smote_lr, X_train, y_train, X_test, y_test, "SMOTE")})

# --- Random Forest ---
pipe_baseline_rf = ImbPipeline([("prep", tree_preprocessor),
                                 ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))])
imbalance_comparison.append({"Model": "Random Forest",
                              **evaluate_quick(pipe_baseline_rf, X_train, y_train, X_test, y_test, "Baseline")})

pipe_cw_rf = ImbPipeline([("prep", tree_preprocessor),
                           ("model", RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1))])
imbalance_comparison.append({"Model": "Random Forest",
                              **evaluate_quick(pipe_cw_rf, X_train, y_train, X_test, y_test, "class_weight='balanced'")})

pipe_smote_rf = ImbPipeline([("prep", tree_preprocessor),
                              ("smote", SMOTE(random_state=RANDOM_STATE)),
                              ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))])
imbalance_comparison.append({"Model": "Random Forest",
                              **evaluate_quick(pipe_smote_rf, X_train, y_train, X_test, y_test, "SMOTE")})

imbalance_df = pd.DataFrame(imbalance_comparison)
imbalance_df = imbalance_df[["Model", "Strategy", "Accuracy", "Recall (Approved)",
                              "Precision (Approved)", "F1 (Approved)", "ROC-AUC"]]

imbalance_df.round(4)

,Model,Strategy,Accuracy,Recall (Approved),Precision (Approved),F1 (Approved),ROC-AUC
0,Logistic Regression,Baseline,0.9043,0.7620,0.7983,0.7797,0.9586
1,Logistic Regression,class_weight='balanced',0.8653,0.9260,0.6351,0.7535,0.9584
2,Logistic Regression,SMOTE,0.8714,0.9115,0.6504,0.7591,0.9587
3,Random Forest,Baseline,0.9301,0.7725,0.8988,0.8309,0.9747
4,Random Forest,class_weight='balanced',0.9278,0.7575,0.9018,0.8234,0.9748
5,Random Forest,SMOTE,0.9263,0.7970,0.8612,0.8278,0.9717


In [29]:
plt.figure(figsize=(9, 5))
sns.barplot(data=imbalance_df, x="Model", y="Recall (Approved)", hue="Strategy")
plt.title("Comparison of Imbalance Handling Strategies (Recall for Approved Class)")
plt.tight_layout()
plt.savefig("outputs/02_imbalance_strategy_comparison.png")
plt.show()

In [30]:
def build_pipeline(model, preprocessor, use_smote=True):
    steps = [("preprocessor", preprocessor)]
    if use_smote:
        steps.append(("smote", SMOTE(random_state=RANDOM_STATE)))
    steps.append(("model", model))
    return ImbPipeline(steps)

### Hyperparameter Tuning

In [ ]:
# from sklearn.model_selection import GridSearchCV
# # FOR knn,xgboost,gradient boosting, and random forest

# knn = KNeighborsClassifier()
# xgb = XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss')
# gb = GradientBoostingClassifier(random_state=RANDOM_STATE)
# rf = RandomForestClassifier(random_state=RANDOM_STATE)

# param_grid_knn = {
#     "model__n_neighbors": [5, 9, 13, 17, 21, 31],
#     "model__weights": ["uniform", "distance"],
#     "model__p": [1, 2]
# }
# param_grid_xgb = {
#     "model__n_estimators": [100, 200,300],
#     "model__learning_rate": [0.05, 0.1],
#     "model__max_depth": [3, 5, 7],
#     "model__subsample": [0.8, 1.0],
#     "model__colsample_bytree": [0.8, 1.0]
# }
# param_grid_gb = {
#     "model__n_estimators": [100, 200],
#     "model__learning_rate": [0.05, 0.1],
#     "model__max_depth": [2, 3],
#     "model__subsample": [0.8, 1.0]
# }
# param_grid_rf = {
#     "model__n_estimators": [100, 200],
#     "model__max_depth": [10, 20],
#     "model__min_samples_split": [2, 5],
#     "model__min_samples_leaf": [1, 2],
#     "model__max_features": ["sqrt"]
# }

# best_models = {}

# for model_name, (model, preprocessor) in [
#     ("KNN", (knn, linear_preprocessor)),
#     ("XGBoost", (xgb, tree_preprocessor)),
#     ("Gradient Boosting", (gb, tree_preprocessor)),
#     ("Random Forest", (rf, tree_preprocessor))
# ]:

#     if model_name == "KNN":
#         param_grid = param_grid_knn
#     elif model_name == "XGBoost":
#         param_grid = param_grid_xgb
#     elif model_name == "Gradient Boosting":
#         param_grid = param_grid_gb
#     elif model_name == "Random Forest":
#         param_grid = param_grid_rf

#     pipeline = build_pipeline(
#         model,
#         preprocessor,
#         use_smote=False
#     )

#     grid_search = GridSearchCV(
#         pipeline,
#         param_grid,
#         cv=5,
#         scoring="f1",
#         n_jobs=-1,
#         refit=True
#     )

#     grid_search.fit(X_train, y_train)

#     best_models[model_name] = grid_search.best_estimator_

#     print(f"Best Parameters for {model_name}:")
#     print(grid_search.best_params_)

#     print(f"Best CV F1 for {model_name}:")
#     print(grid_search.best_score_)

Best Parameters for KNN:
{'model__n_neighbors': 17, 'model__p': 1, 'model__weights': 'distance'}
Best CV F1 for KNN:
0.7367299868070367
Best Parameters for XGBoost:
{'model__colsample_bytree': 1.0, 'model__learning_rate': 0.1, 'model__max_depth': 7, 'model__n_estimators': 300, 'model__subsample': 1.0}
Best CV F1 for XGBoost:
0.8423284414040533
Best Parameters for Gradient Boosting:
{'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200, 'model__subsample': 0.8}
Best CV F1 for Gradient Boosting:
0.8254798025513175
Best Parameters for Random Forest:
{'model__max_depth': 20, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 2, 'model__min_samples_split': 5, 'model__n_estimators': 100}
Best CV F1 for Random Forest:
0.8235911414655955


In [66]:
model_configs = {
    "Logistic Regression": (LogisticRegression(max_iter=1000, random_state=42), linear_preprocessor),
    # "KNN": (KNeighborsClassifier(n_neighbors=5, weights="distance", p=2), linear_preprocessor),
    # "SVC": (SVC(random_state=42,probability=True) ,linear_preprocessor),
    "Random Forest": (RandomForestClassifier( n_estimators=200,max_depth=10,min_samples_split=4,min_samples_leaf=4,random_state=42), tree_preprocessor),
    "Gradient Boosting": (GradientBoostingClassifier(learning_rate=0.05, max_depth=3,n_estimators=200,random_state=42), tree_preprocessor),
    "XGBoost": (   XGBClassifier( n_estimators=200, max_depth=5, learning_rate=0.05, min_child_weight=5, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=2,
        random_state=42,eval_metric="logloss"),tree_preprocessor),
    "AdaBoost": (AdaBoostClassifier(random_state=42), tree_preprocessor),
    "Decision Tree": (DecisionTreeClassifier(  max_depth=5, min_samples_split=10, min_samples_leaf=5, random_state=42), tree_preprocessor)
}

In [67]:
pipelines_without_smote = {name: build_pipeline(model, prep, use_smote=False) for name, (model, prep) in model_configs.items()}
SVC_TRAIN_SAMPLE_SIZE = 8000

results_without_smote = []
roc_data_without_smote = {}
fitted_pipelines_without_smote = {}

for name, pipe in pipelines_without_smote.items():
    if name == "SVC" and len(X_train) > SVC_TRAIN_SAMPLE_SIZE:
        X_train_fit, _, y_train_fit, _ = train_test_split(
            X_train, y_train, train_size=SVC_TRAIN_SAMPLE_SIZE,
            random_state=RANDOM_STATE, stratify=y_train
        )
        print(f"[{name}] training on a {SVC_TRAIN_SAMPLE_SIZE}-row stratified subsample for speed")
    else:
        X_train_fit, y_train_fit = X_train, y_train

    pipe.fit(X_train_fit, y_train_fit)
    fitted_pipelines_without_smote[name] = pipe

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    y_train_pred = pipe.predict(X_train_fit)
    y_train_proba = pipe.predict_proba(X_train_fit)[:, 1]

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    cv_scores = cross_val_score(pipe, X_train_fit, y_train_fit, cv=cv, scoring="f1", n_jobs=-1)

    results_without_smote.append({
        "Model": name,
        
            # Test metrics
            "Test Accuracy": accuracy_score(y_test, y_pred),
            "Test Precision": precision_score(y_test, y_pred),
            "Test Recall": recall_score(y_test, y_pred),
            "Test F1": f1_score(y_test, y_pred),
            "Test ROC-AUC": roc_auc_score(y_test, y_proba),
        
           # Train metrics
            "Train Accuracy": accuracy_score(y_train_fit, y_train_pred),
            "Train Precision": precision_score(y_train_fit, y_train_pred),
            "Train Recall": recall_score(y_train_fit, y_train_pred),
            "Train F1": f1_score(y_train_fit, y_train_pred),
        
        
            # Cross-validation
            "CV F1 (mean)": cv_scores.mean(),
            "CV F1 (std)": cv_scores.std(),
    })

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_data_without_smote[name] = (fpr, tpr, roc_auc_score(y_test, y_proba))

    print(f"\n===== {name} =====")
    print(classification_report(y_test, y_pred, target_names=["Rejected", "Approved"]))
    print(classification_report(y_train_fit, y_train_pred, target_names=["Rejected", "Approved"]))
    print(confusion_matrix(y_test, y_pred))

results_df_without_smote = (
    pd.DataFrame(results_without_smote)
    .sort_values("Test F1", ascending=False)
    .reset_index(drop=True)
)
results_df_without_smote.round(4)


===== Logistic Regression =====
              precision    recall  f1-score   support

    Rejected       0.93      0.94      0.94      7000
    Approved       0.80      0.76      0.78      2000

    accuracy                           0.90      9000
   macro avg       0.87      0.85      0.86      9000
weighted avg       0.90      0.90      0.90      9000

              precision    recall  f1-score   support

    Rejected       0.93      0.94      0.94     28000
    Approved       0.79      0.76      0.77      8000

    accuracy                           0.90     36000
   macro avg       0.86      0.85      0.85     36000
weighted avg       0.90      0.90      0.90     36000

[[6615  385]
 [ 476 1524]]

===== Random Forest =====
              precision    recall  f1-score   support

    Rejected       0.93      0.98      0.95      7000
    Approved       0.91      0.73      0.81      2000

    accuracy                           0.92      9000
   macro avg       0.92      0.85      0.

,Model,Test Accuracy,Test Precision,Test Recall,Test F1,Test ROC-AUC,Train Accuracy,Train Precision,Train Recall,Train F1,CV F1 (mean),CV F1 (std)
0,XGBoost,0.9296,0.8894,0.7800,0.8311,0.9753,0.9336,0.8989,0.7900,0.8409,0.8231,0.0095
1,Gradient Boosting,0.9223,0.8711,0.7635,0.8137,0.9719,0.9244,0.8759,0.7686,0.8188,0.8112,0.0087
2,Random Forest,0.9232,0.9083,0.7280,0.8082,0.9705,0.9311,0.9301,0.7462,0.8281,0.8053,0.0078
3,Logistic Regression,0.9043,0.7983,0.7620,0.7797,0.9586,0.8999,0.7852,0.7566,0.7706,0.7680,0.0055
4,AdaBoost,0.9042,0.8293,0.7165,0.7688,0.9572,0.9029,0.8258,0.7134,0.7655,0.7717,0.0088
5,Decision Tree,0.9046,0.8627,0.6785,0.7596,0.9520,0.9074,0.8723,0.6831,0.7662,0.7619,0.0058


In [68]:
results_df_without_smote.round(4).to_csv("outputs/03_model_results_without_smote.csv", index=False)

In [69]:
plt.figure(figsize=(9, 6))
order = results_df_without_smote.sort_values("Train F1")
plt.barh(order["Model"], order["Train F1"], color="#2e8b57")
for i, v in enumerate(order["Train F1"]):
    plt.text(v + 0.005, i, f"{v:.3f}", va="center")
plt.xlabel("F1 Score (Approved class)")
plt.title("Comparison of Models — F1 Score")
plt.tight_layout()
plt.savefig("outputs/03_models_f1_comparison.png")
plt.show()

In [71]:
plt.figure(figsize=(8, 7))
for name, (fpr, tpr, auc) in roc_data_without_smote.items():
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Comparison of ROC Curves between All Models")
plt.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.savefig("outputs/04_roc_curves_comparison.png")
plt.show()

In [72]:
best_model_name = results_df_without_smote.iloc[0]["Model"]
best_pipeline = fitted_pipelines_without_smote[best_model_name]
print("Best model (by F1):", best_model_name)

y_pred_best = best_pipeline.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(5.5, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Rejected", "Approved"], yticklabels=["Rejected", "Approved"])
plt.title(f"Confusion Matrix — {best_model_name}")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig("outputs/05_best_model_confusion_matrix.png")
plt.show()

Best model (by F1): XGBoost


In [73]:
import os
os.makedirs("deployment", exist_ok=True)
joblib.dump(best_pipeline, "deployment/loan_pipeline.pkl")

with open("deployment/best_model_info.txt", "w", encoding="utf-8") as f:
    f.write(f"Best model: {best_model_name}\n")
    f.write(results_df_without_smote.iloc[0].to_string())

print("Saved deployment/loan_pipeline.pkl")
print(results_df_without_smote.iloc[0])

Saved deployment/loan_pipeline.pkl
Model               XGBoost
Test Accuracy      0.929556
Test Precision     0.889396
Test Recall            0.78
Test F1            0.831113
Test ROC-AUC       0.975331
Train Accuracy     0.933583
Train Precision    0.898876
Train Recall           0.79
Train F1           0.840929
CV F1 (mean)       0.823101
CV F1 (std)        0.009509
Name: 0, dtype: object


In [79]:
sample = X_test.sample(1, random_state=42)

print(sample)

actual = y_test.loc[sample.index].iloc[0]

print("Actual:", actual)

      person_age person_gender person_education  person_income  \
3085        24.0          male         Bachelor        93646.0   

      person_emp_exp person_home_ownership  loan_amnt loan_intent  \
3085               0                  RENT     3350.0    PERSONAL   

      loan_int_rate  loan_percent_income  cb_person_cred_hist_length  \
3085          11.01                 0.04                         3.0   

      credit_score previous_loan_defaults_on_file  loan_burden  \
3085           619                            Yes       0.4404   

      total_interest_cost  income_per_exp_year  
3085              368.835              93646.0  
Actual: 0


In [84]:
sample = X_test.sample(3, random_state=3)
for idx, row in sample.iterrows():
    actual = y_test.loc[idx]
    print(row)
    print(f"Sample index: {idx}, Actual: {actual}")

person_age                             23.0
person_gender                        female
person_education                   Bachelor
person_income                      102776.0
person_emp_exp                            0
person_home_ownership              MORTGAGE
loan_amnt                            8000.0
loan_intent                       EDUCATION
loan_int_rate                          7.14
loan_percent_income                    0.08
cb_person_cred_hist_length              2.0
credit_score                            649
previous_loan_defaults_on_file          Yes
loan_burden                          0.5712
total_interest_cost                   571.2
income_per_exp_year                102776.0
Name: 12861, dtype: object
Sample index: 12861, Actual: 0
person_age                            34.0
person_gender                       female
person_education                    Master
person_income                      73402.0
person_emp_exp                          13
person_home_ownership  

In [74]:
import sys
import sklearn
import imblearn
import pandas
import numpy
import joblib
import xgboost

print("Python:", sys.executable)
print("sklearn:", sklearn.__version__)
print("imblearn:", imblearn.__version__)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("joblib:", joblib.__version__)
print("xgboost:", xgboost.__version__)

Python: f:\anaconda3\python.exe
sklearn: 1.6.1
imblearn: 0.13.0
pandas: 2.2.3
numpy: 2.4.6
joblib: 1.4.2
xgboost: 3.1.2


### Handling imbalanced class

In [70]:
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve
)

results = []
roc_data = {}
fitted_pipelines = {}

SVC_TRAIN_SAMPLE_SIZE = 8000

for name, (model, preprocessor) in model_configs.items():

    if name == "SVC" and len(X_train) > SVC_TRAIN_SAMPLE_SIZE:
        X_train_fit, _, y_train_fit, _ = train_test_split(
            X_train, y_train,
            train_size=SVC_TRAIN_SAMPLE_SIZE,
            random_state=RANDOM_STATE,
            stratify=y_train
        )
    else:
        X_train_fit = X_train
        y_train_fit = y_train

    pipe = build_pipeline(
        clone(model),
        clone(preprocessor),
        use_smote=True
    )

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    cv_scores = cross_val_score(
        pipe,
        X_train_fit,
        y_train_fit,
        cv=cv,
        scoring="f1",
        n_jobs=-1
    )

    pipe.fit(X_train_fit, y_train_fit)

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    fitted_preprocessor = pipe.named_steps["preprocessor"]
    fitted_smote = pipe.named_steps["smote"]
    fitted_model = pipe.named_steps["model"]

    X_train_processed = fitted_preprocessor.transform(X_train_fit)

    X_train_smote, y_train_smote = fitted_smote.fit_resample(
        X_train_processed,
        y_train_fit
    )

    y_train_pred = fitted_model.predict(X_train_smote)

    results.append({
        "Model": name,
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test Precision": precision_score(y_test, y_pred),
        "Test Recall": recall_score(y_test, y_pred),
        "Test F1": f1_score(y_test, y_pred),
        "Test ROC-AUC": roc_auc_score(y_test, y_proba),
        "Train Accuracy": accuracy_score(y_train_smote, y_train_pred),
        "Train Precision": precision_score(y_train_smote, y_train_pred),
        "Train Recall": recall_score(y_train_smote, y_train_pred),
        "Train F1": f1_score(y_train_smote, y_train_pred),
        "CV F1 (mean)": cv_scores.mean(),
        "CV F1 (std)": cv_scores.std()
    })

    fpr, tpr, _ = roc_curve(y_test, y_proba)

    roc_data[name] = (
        fpr,
        tpr,
        roc_auc_score(y_test, y_proba)
    )

    print(f"\n===== {name} =====")
    print("\nTraining data:")
    print("Before SMOTE:", X_train_fit.shape)
    print("After SMOTE:", X_train_smote.shape)

    print("\nClass distribution BEFORE SMOTE:")
    print(pd.Series(y_train_fit).value_counts())

    print("\nClass distribution AFTER SMOTE:")
    print(pd.Series(y_train_smote).value_counts())

    print("\nClassification Report:")
    print(classification_report(
        y_test,
        y_pred,
        target_names=["Rejected", "Approved"]
    ))

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    fitted_pipelines[name] = pipe

results_df = (
    pd.DataFrame(results)
    .sort_values("Test F1", ascending=False)
    .reset_index(drop=True)
)

print("\n=== SMOTE Results (Train metrics calculated on SMOTE-resampled training data) ===")
results_df.round(4)



===== Logistic Regression =====

Training data:
Before SMOTE: (36000, 16)
After SMOTE: (56000, 25)

Class distribution BEFORE SMOTE:
loan_status
0    28000
1     8000
Name: count, dtype: int64

Class distribution AFTER SMOTE:
loan_status
0    28000
1    28000
Name: count, dtype: int64

Classification Report:
              precision    recall  f1-score   support

    Rejected       0.97      0.86      0.91      7000
    Approved       0.65      0.91      0.76      2000

    accuracy                           0.87      9000
   macro avg       0.81      0.89      0.84      9000
weighted avg       0.90      0.87      0.88      9000

Confusion Matrix:
[[6020  980]
 [ 177 1823]]

===== Random Forest =====

Training data:
Before SMOTE: (36000, 16)
After SMOTE: (56000, 25)

Class distribution BEFORE SMOTE:
loan_status
0    28000
1     8000
Name: count, dtype: int64

Class distribution AFTER SMOTE:
loan_status
0    28000
1    28000
Name: count, dtype: int64

Classification Report:
            

,Model,Test Accuracy,Test Precision,Test Recall,Test F1,Test ROC-AUC,Train Accuracy,Train Precision,Train Recall,Train F1,CV F1 (mean),CV F1 (std)
0,XGBoost,0.9241,0.8501,0.7995,0.8240,0.9726,0.9525,0.9621,0.9421,0.9520,0.8220,0.0083
1,Gradient Boosting,0.9167,0.8131,0.8115,0.8123,0.9690,0.9439,0.9452,0.9426,0.9439,0.8093,0.0059
2,Random Forest,0.9133,0.7924,0.8265,0.8091,0.9674,0.9461,0.9426,0.9501,0.9463,0.8039,0.0069
3,AdaBoost,0.8924,0.7279,0.8240,0.7730,0.9542,0.9225,0.9149,0.9316,0.9232,0.7618,0.0058
4,Logistic Regression,0.8714,0.6504,0.9115,0.7591,0.9587,0.8928,0.8661,0.9292,0.8966,0.7546,0.0028
5,Decision Tree,0.8859,0.7206,0.7945,0.7558,0.9459,0.8964,0.9083,0.8819,0.8949,0.7591,0.0064
